In [1]:
# ============================================================
# PHASE 1 — DATA PIPELINE & FEATURE ENGINEERING
# Run: python phase1.py
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import pickle
import re
import warnings
warnings.filterwarnings('ignore')

# ── Load Data ────────────────────────────────────────────────
df = pd.read_csv('delivery_data.csv')
print(f"Shape: {df.shape}")
print(f"Route types:\n{df['route_type'].value_counts()}")
print(f"Unique trips: {df['trip_uuid'].nunique()}")

# ── Parse Timestamps ─────────────────────────────────────────
df['trip_creation_time'] = pd.to_datetime(df['trip_creation_time'], errors='coerce')
df['od_start_time']      = pd.to_datetime(df['od_start_time'],      errors='coerce')
df['od_end_time']        = pd.to_datetime(df['od_end_time'],        errors='coerce')

df['creation_hour']      = df['trip_creation_time'].dt.hour
df['creation_dayofweek'] = df['trip_creation_time'].dt.dayofweek
df['creation_month']     = df['trip_creation_time'].dt.month
df['is_weekend']         = df['creation_dayofweek'].isin([5,6]).astype(int)

# ── Parse Facility Names ──────────────────────────────────────
def parse_facility_name(name_str):
    if pd.isna(name_str):
        return pd.Series({'facility_name':None,'state':None,
                          'city':None,'facility_type':None})
    state_match   = re.search(r'\(([^)]+)\)', name_str)
    state         = state_match.group(1).strip() if state_match else None
    facility_name = re.sub(r'\s*\([^)]*\)', '', name_str).strip()
    parts         = facility_name.split('_')
    city          = parts[0].strip() if len(parts) >= 1 else facility_name
    facility_type = parts[-1].strip() if len(parts) > 1 else None
    return pd.Series({'facility_name':facility_name,'state':state,
                      'city':city,'facility_type':facility_type})

src_parsed = df['source_name'].apply(parse_facility_name).add_prefix('src_')
dst_parsed = df['destination_name'].apply(parse_facility_name).add_prefix('dst_')
df = pd.concat([df, src_parsed, dst_parsed], axis=1)

# ── Delay Feature ─────────────────────────────────────────────
df['_check']   = df['segment_actual_time'] / df['segment_osrm_time'].replace(0, np.nan)
match_rate     = (df['_check'].round(4) == df['segment_factor'].round(4)).mean()
print(f"segment_factor verification: {match_rate:.1%}")
df.drop(columns=['_check'], inplace=True)

df['delay_ratio'] = df['segment_factor']
before = len(df)
df = df[(df['delay_ratio'] > 0) & (df['delay_ratio'] < 20)].copy()
print(f"Rows removed: {before - len(df)} | Kept: {len(df)}")

# ── Thresholds ────────────────────────────────────────────────
percentiles = {p: np.percentile(df['delay_ratio'], p)
               for p in [50, 75, 90, 95, 99]}
THRESHOLD_MODERATE = percentiles[75]
THRESHOLD_SEVERE   = percentiles[90]
THRESHOLD_EXTREME  = percentiles[95]

df['is_moderate_delay'] = (df['delay_ratio'] > THRESHOLD_MODERATE).astype(int)
df['is_severe_delay']   = (df['delay_ratio'] > THRESHOLD_SEVERE).astype(int)
df['is_extreme_delay']  = (df['delay_ratio'] > THRESHOLD_EXTREME).astype(int)

print(f"P50={percentiles[50]:.3f} P75={THRESHOLD_MODERATE:.3f} "
      f"P90={THRESHOLD_SEVERE:.3f} P95={THRESHOLD_EXTREME:.3f}")

# ── Speed Features ────────────────────────────────────────────
df['seg_speed_actual']  = (df['segment_osrm_distance'] /
                           (df['segment_actual_time']/60).replace(0, np.nan))
df['seg_speed_osrm']    = (df['segment_osrm_distance'] /
                           (df['segment_osrm_time']/60).replace(0, np.nan))
df['speed_efficiency']  = df['seg_speed_actual'] / df['seg_speed_osrm'].replace(0, np.nan)

# ── Hypothesis Test ───────────────────────────────────────────
ftl     = df[df['route_type']=='FTL']['delay_ratio']
carting = df[df['route_type']=='Carting']['delay_ratio']
stat, pval = stats.mannwhitneyu(ftl, carting, alternative='two-sided')
pooled_std = np.sqrt((ftl.std()**2 + carting.std()**2) / 2)
cohens_d   = (ftl.mean() - carting.mean()) / pooled_std
print(f"\nFTL median: {ftl.median():.3f}  Carting median: {carting.median():.3f}")
print(f"p-value: {pval:.2e}  Cohen's d: {cohens_d:.4f}")

# ── Trip Aggregation ──────────────────────────────────────────
trip_agg = df.groupby('trip_uuid').agg(
    route_type          =('route_type',                    'first'),
    source_center       =('source_center',                 'first'),
    destination_center  =('destination_center',            'first'),
    src_state           =('src_state',                     'first'),
    dst_state           =('dst_state',                     'first'),
    src_city            =('src_city',                      'first'),
    src_facility_type   =('src_facility_type',             'first'),
    dst_facility_type   =('dst_facility_type',             'first'),
    total_actual_time   =('segment_actual_time',           'sum'),
    total_osrm_time     =('segment_osrm_time',             'sum'),
    total_distance      =('actual_distance_to_destination','max'),
    total_osrm_distance =('segment_osrm_distance',         'sum'),
    mean_delay_ratio    =('delay_ratio',                   'mean'),
    max_delay_ratio     =('delay_ratio',                   'max'),
    n_severe_segments   =('is_severe_delay',               'sum'),
    n_extreme_segments  =('is_extreme_delay',              'sum'),
    total_segments      =('delay_ratio',                   'count'),
    mean_speed_efficiency=('speed_efficiency',             'mean'),
    creation_hour       =('creation_hour',                 'first'),
    creation_dayofweek  =('creation_dayofweek',            'first'),
    creation_month      =('creation_month',                'first'),
    is_weekend          =('is_weekend',                    'first'),
).reset_index()

trip_agg['trip_delay_ratio']     = (trip_agg['total_actual_time'] /
                                    trip_agg['total_osrm_time'].replace(0, np.nan))
trip_agg['pct_severe_segments']  = trip_agg['n_severe_segments'] / trip_agg['total_segments']
trip_agg['pct_extreme_segments'] = trip_agg['n_extreme_segments'] / trip_agg['total_segments']
trip_agg['is_interstate']        = (trip_agg['src_state'] != trip_agg['dst_state']).astype(int)
trip_agg['is_trip_severe']       = (trip_agg['trip_delay_ratio'] > THRESHOLD_SEVERE).astype(int)
trip_agg['is_trip_extreme']      = (trip_agg['trip_delay_ratio'] > THRESHOLD_EXTREME).astype(int)
trip_agg['route_type_enc']       = (trip_agg['route_type'] == 'FTL').astype(int)

print(f"\ntrip_agg shape: {trip_agg.shape}")
print(trip_agg['trip_delay_ratio'].describe())



Shape: (144867, 24)
Route types:
route_type
FTL        99660
Carting    45207
Name: count, dtype: int64
Unique trips: 14817
segment_factor verification: 98.4%
Rows removed: 3206 | Kept: 141661
P50=1.692 P75=2.250 P90=3.250 P95=4.300

FTL median: 1.667  Carting median: 1.818
p-value: 4.86e-180  Cohen's d: -0.2095

trip_agg shape: (14804, 30)
count    14804.000000
mean         2.262921
std          1.338714
min          0.257732
25%          1.571110
50%          1.921053
75%          2.484848
max         19.421053
Name: trip_delay_ratio, dtype: float64


In [2]:
# ── Save Checkpoint ───────────────────────────────────────────
with open('phase1_checkpoint.pkl', 'wb') as f:
    pickle.dump({
        'df':                 df,
        'trip_agg':           trip_agg,
        'THRESHOLD_MODERATE': THRESHOLD_MODERATE,
        'THRESHOLD_SEVERE':   THRESHOLD_SEVERE,
        'THRESHOLD_EXTREME':  THRESHOLD_EXTREME,
        'percentiles':        percentiles,
    }, f, protocol=4)

print("\n✅ Phase 1 complete — saved phase1_checkpoint.pkl")


✅ Phase 1 complete — saved phase1_checkpoint.pkl
